**NOTE:**
**Data is stored on Google Drive**
**so Mount drive before running**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

path = "/content/drive/MyDrive/minor2/data/minor_wGhats_tiles"
files = os.listdir(path)

print(len(files))
print(files[:5])

In [ ]:
tile_paths = [os.path.join(path, f) for f in files if f.endswith('.tif')]

In [ ]:
import rasterio

sample_path = tile_paths[0]

with rasterio.open(sample_path) as src:
    data = src.read()

print("Shape:", data.shape)

In [ ]:
t1 = data[:6]
t2 = data[6:12]
label = data[12]

print("T1 shape:", t1.shape)
print("T2 shape:", t2.shape)
print("Label shape:", label.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(label)
plt.title("Label (Deforestation)")
plt.colorbar()
plt.show()

In [ ]:
# RGB from T1
rgb = t1[[2,1,0]]  # B4, B3, B2

plt.imshow(rgb.transpose(1,2,0) / 3000)
plt.title("T1 RGB")
plt.show()

In [ ]:
import numpy as np

print("Min:", label.min())
print("Max:", label.max())
print("Unique:", np.unique(label)[:10])

In [ ]:
for path in tile_paths:
    with rasterio.open(path) as src:
        data = src.read()
    
    if not np.isnan(data).all():
        print("VALID TILE:", path)
        break

In [ ]:
data = np.nan_to_num(data, nan=0.0)

In [ ]:
with rasterio.open(sample_path) as src:
    data = src.read()

data = np.nan_to_num(data, nan=0.0)

t1 = data[:6]
t2 = data[6:12]
label = data[12]

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(label, cmap='gray')
plt.title("Label")
plt.show()

In [ ]:
# RGB from T1
rgb = t1[[2,1,0]]  # B4, B3, B2

plt.imshow(rgb.transpose(1,2,0) / 3000)
plt.title("T1 RGB")
plt.show()

In [ ]:
import rasterio
import numpy as np

def load_tile(path):
    with rasterio.open(path) as src:
        data = src.read()

    # Replace NaNs
    data = np.nan_to_num(data, nan=0.0)

    t1 = data[:6]
    t2 = data[6:12]
    label = data[12]

    # Normalize
    t1 = t1 / 10000.0
    t2 = t2 / 10000.0

    return t1, t2, label

In [ ]:
def create_patches(t1, t2, label, size=128):
    patches = []

    _, H, W = t1.shape

    for i in range(0, H - size, size):
        for j in range(0, W - size, size):

            p1 = t1[:, i:i+size, j:j+size]
            p2 = t2[:, i:i+size, j:j+size]
            pl = label[i:i+size, j:j+size]

            # skip empty patches
            if np.sum(pl) == 0:   #gc
                continue  

            patches.append((p1, p2, pl))

    return patches

In [ ]:
import torch
from torch.utils.data import Dataset

class DeforestationDataset(Dataset):
    def __init__(self, tile_paths):
        self.samples = []

        for path in tile_paths:
            t1, t2, label = load_tile(path)

            patches = create_patches(t1, t2, label)

            self.samples.extend(patches)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        t1, t2, label = self.samples[idx]

        x = np.concatenate([t1, t2], axis=0)

        return torch.tensor(x, dtype=torch.float32), \
               torch.tensor(label, dtype=torch.float32).unsqueeze(0)

In [ ]:
dataset = DeforestationDataset(tile_paths)

print("Total samples:", len(dataset))

x, y = dataset[0]

print("Input shape:", x.shape)   # (12, 128, 128)
print("Label shape:", y.shape)  # (1, 128, 128)

In [ ]:
#bas mera dil tha isliye add kiya hai ye codeblock, u can ignore!
import matplotlib.pyplot as plt
import numpy as np

# sample le lo
x, y = dataset[0]

# convert tensors → numpy
x = x.numpy()
y = y.numpy()[0]

# split back
t1 = x[:6]
t2 = x[6:12]

# RGB banate hain
def get_rgb(img):
    rgb = img[[2,1,0]]  # B4, B3, B2
    rgb = rgb.transpose(1,2,0)

    # normalize for display
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    return rgb

rgb_t1 = get_rgb(t1)
rgb_t2 = get_rgb(t2)

# plot
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(rgb_t1)
plt.title("T1 (Before)")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(rgb_t2)
plt.title("T2 (After)")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(y, cmap='gray')
plt.title("Label (Deforestation)")
plt.axis('off')

plt.show()

In [ ]:
import torch
import torch.nn as nn

class SiameseUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # shared encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(6, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU()
        )

        # decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        t1 = x[:, :6, :, :]
        t2 = x[:, 6:12, :, :]

        f1 = self.encoder(t1)
        f2 = self.encoder(t2)

        diff = torch.abs(f2 - f1)

        out = self.decoder(diff)

        return out

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SiameseUNet().to(device)

criterion = nn.BCEWithLogitsLoss()   # binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        pred = model(x)

        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
model.eval()

x, y = dataset[0]

x = x.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)
    pred = torch.sigmoid(pred)

pred = pred.cpu().numpy()[0][0]

# threshold
binary = (pred > 0.5).astype(float)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(pred, cmap='viridis')
plt.title("Prediction Prob")

plt.subplot(1,2,2)
plt.imshow(binary, cmap='gray')
plt.title("Binary Output")

plt.show()